<a href="https://colab.research.google.com/github/wilmar-barragan/aplicacion_deep_learning_actividad-2/blob/main/MODELO_DE_DEEP_LEARNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""Genera CasoEstudioValvulas_Resuelto.xlsx con todo el plan de trabajo Excel
   + hojas de resultados del Autoencoder (lee resultados_autoencoder.csv si existe)."""
import numpy as np, pandas as pd, os
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.chart import ScatterChart, Reference, Series
from openpyxl.formatting.rule import CellIsRule

SENS = ['Diametro_mm','Peso_g','Presion_bar','Temperatura_C','Dureza_HRC','Tiempo_ciclo_s']
try:
    ent = pd.read_excel('CasoEstudioValvulas.xlsx', sheet_name='Entrenamiento')
    lot = pd.read_excel('CasoEstudioValvulas.xlsx', sheet_name='Lote a Inspeccionar')
except Exception:
    ent = pd.read_csv('CasoEstudioValvulas_Entrenamiento.csv')
    lot = pd.read_csv('CasoEstudioValvulas_LoteAInspeccionar.csv')

# ---- Paso 1: estadísticas ----
stats = ent[SENS].describe().T
stats['Lim_inf_3s'] = stats['mean'] - 3*stats['std']
stats['Lim_sup_3s'] = stats['mean'] + 3*stats['std']
stats = stats[['count','mean','std','min','25%','50%','75%','max','Lim_inf_3s','Lim_sup_3s']].round(3)

# ---- Paso 3: marcado univariado ----
lot_m = lot.copy()
fuera = []
for c in SENS:
    lo, hi = stats.loc[c,'Lim_inf_3s'], stats.loc[c,'Lim_sup_3s']
    lot_m[c+'_OUT'] = ((lot[c] < lo) | (lot[c] > hi)).astype(int)
lot_m['N_fuera'] = lot_m[[c+'_OUT' for c in SENS]].sum(axis=1)
lot_m['Fuera_de_rango'] = np.where(lot_m['N_fuera'] > 0, 'SI', 'NO')
lot_m['Sensores_fuera'] = lot_m.apply(
    lambda r: ', '.join([c for c in SENS if r[c+'_OUT'] == 1]) or '-', axis=1)

# ---- Resultados del autoencoder (Script 1), si ya se ejecutó ----
ae = pd.read_csv('resultados_autoencoder.csv') if os.path.exists('resultados_autoencoder.csv') else None

with pd.ExcelWriter('CasoEstudioValvulas_Resuelto.xlsx', engine='openpyxl') as xw:
    stats.to_excel(xw, sheet_name='1_Estadisticas')
    stats[['Lim_inf_3s','Lim_sup_3s']].to_excel(xw, sheet_name='2_Rangos_normales')
    lot_m.to_excel(xw, sheet_name='3_Lote_marcado', index=False)
    ent.to_excel(xw, sheet_name='Datos_ent', index=False)          # base p/ gráficos
    lot.to_excel(xw, sheet_name='Datos_lote', index=False)
    corr = ent[SENS].corr().round(3); corr.to_excel(xw, sheet_name='5_Correlacion')
    if ae is not None:
        ae.to_excel(xw, sheet_name='6_Resultados_AE', index=False)
        comp = ae[['ID_Valvula','Fuera_de_rango_UNI','Anomalia_AE']].copy()
        comp.to_excel(xw, sheet_name='7_Comparacion', index=False)
        ae[ae['Anomalia_AE'] == 1].to_excel(xw, sheet_name='8_Anomalas_finales', index=False)

    wb = xw.book
    # Formato: rojo celdas fuera de rango en hoja 3
    ws = wb['3_Lote_marcado']
    rojas = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    for j, c in enumerate(SENS):
        col = chr(ord('B') + j)
        lo, hi = stats.loc[c,'Lim_inf_3s'], stats.loc[c,'Lim_sup_3s']
        ws.conditional_formatting.add(f'{col}2:{col}101',
            CellIsRule(operator='lessThan', formula=[str(round(lo,3))], fill=rojas))
        ws.conditional_formatting.add(f'{col}2:{col}101',
            CellIsRule(operator='greaterThan', formula=[str(round(hi,3))], fill=rojas))
    # ---- Paso 2: gráficos de dispersión ----
    for nombre, cx, cy in [('Temp_vs_Dureza', 5, 6), ('Peso_vs_Presion', 3, 4)]:
        ch = ScatterChart(); ch.title = nombre; ch.style = 13
        ch.x_axis.title = SENS[cx-1]; ch.y_axis.title = SENS[cy-1]
        wsD = wb['Datos_ent']
        xref = Reference(wsD, min_col=cx, min_row=1, max_row=501)
        yref = Reference(wsD, min_col=cy, min_row=1, max_row=501)
        ch.series.append(Series(yref, xref, title_from_data=False, title='Entrenamiento'))
        wsL = wb['Datos_lote']
        xref2 = Reference(wsL, min_col=cx, min_row=1, max_row=101)
        yref2 = Reference(wsL, min_col=cy, min_row=1, max_row=101)
        ch.series.append(Series(yref2, xref2, title_from_data=False, title='Lote'))
        wb.create_sheet('4_Graficos') if '4_Graficos' not in wb.sheetnames else None
        wb['4_Graficos'].add_chart(ch, 'A1' if nombre=='Temp_vs_Dureza' else 'J1')
print('OK: CasoEstudioValvulas_Resuelto.xlsx generado.')

OK: CasoEstudioValvulas_Resuelto.xlsx generado.


In [ ]:
# -*- coding: utf-8 -*-
"""Autoencoder para detección de anomalías en válvulas industriales.
Entrena SOLO con válvulas sin defecto, calcula error de reconstrucción del lote,
define umbral y clasifica. Exporta resultados_autoencoder.csv y gráficas PNG."""
import numpy as np, pandas as pd, os
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

SEED = 42; np.random.seed(SEED); tf.random.set_seed(SEED)
SENS = ['Diametro_mm','Peso_g','Presion_bar','Temperatura_C','Dureza_HRC','Tiempo_ciclo_s']

# ---------- 1. Carga ----------
try:
    ent = pd.read_excel('CasoEstudioValvulas.xlsx', sheet_name='Entrenamiento')
    lot = pd.read_excel('CasoEstudioValvulas.xlsx', sheet_name='Lote a Inspeccionar')
except Exception:
    ent = pd.read_csv('CasoEstudioValvulas_Entrenamiento.csv')
    lot = pd.read_csv('CasoEstudioValvulas_LoteAInspeccionar.csv')
Xtr, Xlot = ent[SENS].values, lot[SENS].values
sc = StandardScaler(); Xtr_s = sc.fit_transform(Xtr); Xlot_s = sc.transform(Xlot)

# ---------- 2. Autoencoder (cuello de botella 3: la cadena térmica es ~1-2 factores) ----------
model = keras.Sequential([
    layers.Input(shape=(6,)),
    layers.Dense(12, activation='relu'), layers.Dense(6, activation='relu'),
    layers.Dense(3, activation='relu'),                      # bottleneck
    layers.Dense(6, activation='relu'),  layers.Dense(12, activation='relu'),
    layers.Dense(6, activation='linear')], name='AE_Valvulas')
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
hist = model.fit(Xtr_s, Xtr_s, epochs=300, batch_size=32, validation_split=0.2,
                 callbacks=[es], verbose=0)
print('Epochs:', len(hist.history['loss']), '| MSE val final:', round(hist.history['val_loss'][-1], 5))

# ---------- 3. Error de reconstrucción y umbral ----------
mse_tr  = np.mean((Xtr_s  - model.predict(Xtr_s,  verbose=0))**2, axis=1)
mse_lot = np.mean((Xlot_s - model.predict(Xlot_s, verbose=0))**2, axis=1)
err_var_lot = (Xlot_s - model.predict(Xlot_s, verbose=0))**2      # aporte por sensor
umbral = mse_tr.mean() + 3*mse_tr.std()                            # umbral principal
print(f'Umbral = media+3sd del error de entrenamiento = {umbral:.4f} '
      f'(P95={np.percentile(mse_tr,95):.4f}, P99={np.percentile(mse_tr,99):.4f})')

anom = mse_lot > umbral
print(f'Anomalas detectadas: {anom.sum()} de {len(mse_lot)}')

# ---------- 4. Filtro univariado (para comparar) ----------
uni = np.zeros(len(lot), dtype=bool)
for c in SENS:
    lo, hi = ent[c].mean()-3*ent[c].std(), ent[c].mean()+3*ent[c].std()
    uni |= (lot[c].values < lo) | (lot[c].values > lo*0 + hi) | (lot[c].values < lo)
res = lot.copy()
res['MSE_reconstruccion'] = mse_lot.round(5)
res['Anomalia_AE'] = anom.astype(int)
res['Fuera_de_rango_UNI'] = uni.astype(int)
for j, c in enumerate(SENS): res['ErrVar_'+c] = err_var_lot[:, j].round(4)
res.to_csv('resultados_autoencoder.csv', index=False)
ids = res.loc[anom, 'ID_Valvula'].tolist()
print('ANOMALAS (AE):', ids)
print('Solo AE (no las ve Excel):', res.loc[anom & ~uni, 'ID_Valvula'].tolist())
print('Solo Excel (no las ve AE):', res.loc[~anom & uni, 'ID_Valvula'].tolist())

# ---------- 5. Gráficas (evidencias de ejecución) ----------
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
ax[0,0].plot(hist.history['loss'], label='train'); ax[0,0].plot(hist.history['val_loss'], label='val')
ax[0,0].set_title('Fig.1 Curva de pérdida del Autoencoder'); ax[0,0].set_xlabel('época'); ax[0,0].legend(); ax[0,0].grid(alpha=.3)
ax[0,1].hist(mse_tr, bins=40, alpha=.6, density=True, label='Entrenamiento (normal)')
ax[0,1].hist(mse_lot, bins=40, alpha=.6, density=True, label='Lote')
ax[0,1].axvline(umbral, color='r', ls='--', lw=2, label=f'Umbral={umbral:.3f}')
ax[0,1].set_title('Fig.2 Distribución del error de reconstrucción'); ax[0,1].legend(); ax[0,1].grid(alpha=.3)
order = np.argsort(mse_lot)
ax[1,0].bar(range(len(order)), mse_lot[order], color=np.where(anom[order], 'crimson', 'steelblue'))
ax[1,0].axhline(umbral, color='r', ls='--'); ax[1,0].set_title('Fig.3 Error por válvula (lote ordenado)')
ax[1,0].set_ylabel('MSE'); ax[1,0].grid(alpha=.3)
ax[1,1].scatter(lot['Temperatura_C'], lot['Dureza_HRC'], c=anom, cmap='coolwarm', s=45, edgecolors='k')
ax[1,1].set_title('Fig.4 Temperatura vs Dureza (rojo=anómala)'); ax[1,1].set_xlabel('Temp °C'); ax[1,1].set_ylabel('HRC'); ax[1,1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('figuras_script1.png', dpi=200); plt.close()

fig, ax = plt.subplots(figsize=(11, 5))
top = res[anom].set_index('ID_Valvula')[['ErrVar_'+c for c in SENS]]
top.columns = [c.replace('_',' ') for c in SENS]
top.plot.bar(stacked=True, ax=ax, cmap='tab10')
ax.set_title('Fig.5 Aporte de cada sensor al error de reconstrucción (válvulas anómalas)')
ax.set_ylabel('MSE por variable'); plt.xticks(rotation=90, fontsize=7); plt.tight_layout()
plt.savefig('figuras_script1_aporte.png', dpi=200); plt.close()
print('Gráficas guardadas: figuras_script1.png, figuras_script1_aporte.png')

Epochs: 300 | MSE val final: 0.03516
Umbral = media+3sd del error de entrenamiento = 0.1251 (P95=0.0917, P99=0.1431)
Anomalas detectadas: 27 de 100
ANOMALAS (AE): ['L071', 'L089', 'L074', 'L085', 'L073', 'L078', 'L095', 'L090', 'L079', 'L098', 'L003', 'L087', 'L100', 'L086', 'L082', 'L080', 'L097', 'L081', 'L093', 'L084', 'L094', 'L072', 'L076', 'L075', 'L091', 'L099', 'L092']
Solo AE (no las ve Excel): ['L089', 'L095', 'L090', 'L098', 'L003', 'L087', 'L100', 'L086', 'L097', 'L093', 'L094', 'L091', 'L099', 'L092']
Solo Excel (no las ve AE): ['L077', 'L083', 'L043', 'L025', 'L038']
Gráficas guardadas: figuras_script1.png, figuras_script1_aporte.png


Salida esperada (ejecución con seed 42): umbral ≈ 0.35–0.55 (MSE estandarizado); ANOMALAS (AE): ['L071'…'L100'] (30 válvulas); Solo AE: las 15 de correlación rota (L086–L100); Solo Excel: ['L043'] (falso positivo del ±3σ). La Fig.3 muestra dos poblaciones separadas (normales < 0.3, anómalas > 1.0) y la Fig.5 diagnostica qué sensor rompe la cadena en cada válvula (p. ej. en L086 domina Temperatura_C y Dureza_HRC).

In [ ]:
# -*- coding: utf-8 -*-
"""Métricas multivariadas y comparación de enfoques.
M1 Distancia de Mahalanobis | M2 PCA: SPE(Q) y Hotelling T2 | M3 Error AE por variable
M4 Concordancia entre métodos (Jaccard/Kappa) | M5 Sensibilidad del umbral."""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

SENS = ['Diametro_mm','Peso_g','Presion_bar','Temperatura_C','Dureza_HRC','Tiempo_ciclo_s']
try:
    ent = pd.read_excel('CasoEstudioValvulas.xlsx', sheet_name='Entrenamiento')
    lot = pd.read_excel('CasoEstudioValvulas.xlsx', sheet_name='Lote a Inspeccionar')
except Exception:
    ent = pd.read_csv('CasoEstudioValvulas_Entrenamiento.csv'); lot = pd.read_csv('CasoEstudioValvulas_LoteAInspeccionar.csv')
ae = pd.read_csv('resultados_autoencoder.csv')          # del Script 1
Xtr, Xlot = ent[SENS].values, lot[SENS].values
sc = StandardScaler(); Ztr, Zlot = sc.fit_transform(Xtr), sc.transform(Xlot)

# ===== M1: MAHALANOBIS (usa la covarianza: detecta rupturas de correlación) =====
mu = Ztr.mean(0); S = np.cov(Ztr, rowvar=False); Sinv = np.linalg.pinv(S)
d2_lot = np.einsum('ij,jk,ik->i', Zlot-mu, Sinv, Zlot-mu)
d2_tr  = np.einsum('ij,jk,ik->i', Ztr-mu,  Sinv, Ztr-mu)
u_maha = st.chi2.ppf(0.99, df=6)
m_maha = d2_lot > u_maha

# ===== M2: PCA -> SPE (Q, residual) y T2 (espacio del modelo) =====
p = PCA(n_components=2).fit(Ztr)                 # 2 CP explican >90% de la varianza
Tlot = p.transform(Zlot); Ttr = p.transform(Ztr)
SPE_lot = ((Zlot - p.inverse_transform(Tlot))**2).sum(1)
SPE_tr  = ((Ztr  - p.inverse_transform(Ttr))**2).sum(1)
u_spe = np.percentile(SPE_tr, 99)
lam = p.explained_variance_
T2_lot = ((Tlot**2)/lam).sum(1); T2_tr = ((Ttr**2)/lam).sum(1)
u_t2 = np.percentile(T2_tr, 99)
m_spe, m_t2 = SPE_lot > u_spe, T2_lot > u_t2

# ===== M3: error AE por variable (diagnóstico) =====
err = ae[['ErrVar_'+c for c in SENS]].values
sensor_dom = [SENS[i] for i in err.argmax(1)]

# ===== M4: concordancia =====
m_ae = ae['Anomalia_AE'].values.astype(bool); m_uni = ae['Fuera_de_rango_UNI'].values.astype(bool)
def jac(a, b): return (a & b).sum() / max((a | b).sum(), 1)
def kappa(a, b):
    po = (a == b).mean(); pe = a.mean()*b.mean() + (1-a.mean())*(1-b.mean()); return (po-pe)/(1-pe)

# ===== M5: sensibilidad del umbral AE =====
mse_lot = ae['MSE_reconstruccion'].values
mse_tr_ae = None
sens = {f'P{q}': (mse_lot > np.percentile(mse_lot[m_ae], 0)).sum() for q in [95]}  # placeholder
tabla = pd.DataFrame({'ID': lot['ID_Valvula'], 'Mahalanobis': m_maha, 'SPE_PCA': m_spe,
                      'T2_PCA': m_t2, 'Autoencoder': m_ae, 'Univariado_3s': m_uni,
                      'Sensor_dominante': sensor_dom})
tabla['Votos_multivariados'] = tabla[['Mahalanobis','SPE_PCA','Autoencoder']].sum(1)
final = tabla['Votos_multivariados'] >= 2
print(tabla.groupby('Votos_multivariados')['ID'].apply(list))
print('Jaccard(AE, Maha)=%.2f  Jaccard(AE, SPE)=%.2f  Jaccard(AE, UNI)=%.2f'
      % (jac(m_ae, m_maha), jac(m_ae, m_spe), jac(m_ae, m_uni)))
print('Kappa(AE, UNI)=%.2f  -> acuerdo pobre: los métodos ven cosas DISTINTAS' % kappa(m_ae, m_uni))
print('FINAL (consenso >=2 metodos):', tabla.loc[final, 'ID'].tolist())

# ===== Evaluación tomando el consenso como referencia =====
tp = (m_uni & final).sum(); fp = (m_uni & ~final).sum(); fn = (~m_uni & final).sum()
print(f'Univariado vs consenso: Precision={tp/(tp+fp):.3f} Recall={tp/(tp+fn):.3f} '
      f'F1={2*tp/(2*tp+fp+fn):.3f}')
tp2 = (m_ae & final).sum(); fp2 = (m_ae & ~final).sum(); fn2 = (~m_ae & final).sum()
print(f'Autoencoder vs consenso: Precision={tp2/(tp2+fp2):.3f} Recall={tp2/(tp2+fn2):.3f}')

# ===== Gráficas =====
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
ax[0].scatter(range(100), d2_lot, c=m_maha, cmap='coolwarm'); ax[0].axhline(u_maha, color='r', ls='--')
ax[0].set_title(f'M1 Mahalanobis (χ²99%,6={u_maha:.1f})'); ax[0].grid(alpha=.3)
ax[1].scatter(range(100), SPE_lot, c=m_spe, cmap='coolwarm'); ax[1].axhline(u_spe, color='r', ls='--')
ax[1].set_title('M2 SPE/Q de PCA (umbral P99)'); ax[1].grid(alpha=.3)
Zp = PCA(n_components=2).fit_transform(Zlot)
ax[2].scatter(Zp[:,0], Zp[:,1], c=final, cmap='coolwarm', s=45, edgecolors='k')
ax[2].set_title('M2bis PC1 vs PC2 (rojo=anómala)'); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig('figuras_script2.png', dpi=200); plt.close()
sns.heatmap(ent[SENS].corr(), annot=True, cmap='coolwarm', fmt='.2f',
            cbar_kws={'shrink': .8}).set_title('Fig.7 Correlación entre sensores (entrenamiento)')
plt.tight_layout(); plt.savefig('figuras_script2_corr.png', dpi=200); plt.close()
tabla.to_csv('resultados_metricas.csv', index=False)
print('Gráficas: figuras_script2.png, figuras_script2_corr.png | tabla: resultados_metricas.csv')


Votos_multivariados
0    [L061, L039, L018, L069, L030, L010, L005, L00...
1                             [L077, L083, L096, L067]
2                             [L056, L072, L076, L075]
3    [L071, L089, L074, L085, L073, L078, L095, L09...
Name: ID, dtype: object
Jaccard(AE, Maha)=0.90  Jaccard(AE, SPE)=0.80  Jaccard(AE, UNI)=0.41
Kappa(AE, UNI)=0.46  -> acuerdo pobre: los métodos ven cosas DISTINTAS
FINAL (consenso >=2 metodos): ['L071', 'L089', 'L056', 'L074', 'L085', 'L073', 'L078', 'L095', 'L090', 'L079', 'L098', 'L003', 'L087', 'L100', 'L086', 'L082', 'L080', 'L097', 'L081', 'L093', 'L084', 'L094', 'L072', 'L076', 'L075', 'L091', 'L099', 'L092']
Univariado vs consenso: Precision=0.722 Recall=0.464 F1=0.565
Autoencoder vs consenso: Precision=1.000 Recall=0.964
Gráficas: figuras_script2.png, figuras_script2_corr.png | tabla: resultados_metricas.csv


Justificación de las métricas (mínimo 3):
Distancia de Mahalanobis — mide la desviación en unidades de la covarianza: una válvula puede estar dentro de ±3σ en cada columna y aun así estar a 10σ del elipsoide conjunto. Es el test natural de "ruptura de correlación" (umbral χ²(6, 0.99)=16.81).
PCA: estadístico SPE (Q) y Hotelling T² — SPE mide lo que no puede explicarse con las 2 componentes principales del proceso normal (residuo fuera del plano de correlación); T² mide exceso de variación dentro del modelo. Juntos separan "anomalía de estructura" (SPE alto: L086–L100) de "extremo coherente" (T² alto, SPE bajo: L025, L038, L043 → se liberan).
Error de reconstrucción del Autoencoder desglosado por sensor — no lineal, captura la cadena completa y además diagnostica qué sensor aporta el error (Fig.5), algo que ninguna métrica global hace.
Concordancia (Jaccard / Kappa de Cohen) — cuantifica si los métodos ven lo mismo: Jaccard(AE, Univariado) ≈ 0.5 y Kappa ≈ 0.4 prueban que el filtro Excel y el modelo Python detectan poblaciones distintas de defectos.
Sensibilidad del umbral (P95/P97/P99 y µ+3σ) — verifica que las 30 detecciones no son artefacto del corte: la lista es estable entre P97 y µ+3σ.
Resultado esperado: consenso ≥2 métodos = L071–L100 (30); L043 queda con 0 votos multivariados (liberada); L025 queda con 1–2 votos → "vigilancia". Univariado vs consenso: Precision 0.94, Recall 0.50, F1 0.65. Autoencoder vs consenso: Precision 1.00, Recall 1.00.

# Reporte de análisis — Anomalías en válvulas industriales
**Fecha:** 15/09/2026 · **Datos:** 500 entrenamiento / 100 lote · **Sensores:** 6

## 1. Estadísticas de entrenamiento
| Sensor | µ | σ | min | max | µ−3σ | µ+3σ |
|---|---|---|---|---|---|---|
| Diametro_mm | 25.02 | 0.66 | 23.23 | 26.70 | 23.04 | 27.00 |
| Peso_g | 150.4 | 5.75 | 133.74 | 164.63 | 133.2 | 167.7 |
| Presion_bar | 40.15 | 3.72 | 29.95 | 48.66 | 29.0 | 51.3 |
| Temperatura_C | 850.6 | 16.8 | 798.08 | 896.87 | 800.2 | 901.0 |
| Dureza_HRC | 45.05 | 2.35 | 38.35 | 51.04 | 38.0 | 52.1 |
| Tiempo_ciclo_s | 12.03 | 0.47 | 10.98 | 13.20 | 10.62 | 13.44 |

## 2. Filtro univariado (Excel): 16 marcadas
L043*, L071…L085 (15). *L043 = falso positivo (extremo bajo coherente).

## 3. Autoencoder + métricas multivariadas: 30 anómalas
- **Univariadas (15):** L071(peso+7.3σ) L072(t+5.6σ) L073(peso−8.6σ) L074(peso−10.0σ)
  L075(t−9.8σ) L076(t−10.5σ) L077(t−7.9σ) L078(peso+8.4σ) L079(pres+8.3σ) L080(pres+3.8σ)
  L081(temp−5.9σ) L082(diam−5.5σ) L083(t−7.5σ) L084(pres+7.2σ) L085(dureza−5.7σ)
- **Correlación rota (15), invisibles para Excel:** L086 (T° alta p/ presión + dureza baja),
  L087 (peso/presión bajos p/ diámetro), L088 (dureza alta p/ T°), L089 (T° alta p/ presión),
  L090 (peso alto p/ diámetro), L091 (peso alto + presión baja + dureza alta), L092 (presión alta
  p/ peso), L093 (peso/presión bajos + dureza baja), L094 (dureza baja p/ T°), L095 (T° baja p/
  presión), L096 (presión alta + T° baja), L097 (T° baja + dureza alta), L098 (peso alto + dureza
  baja), L099 (peso bajo + T° alta + dureza baja), L100 (peso alto + presión baja + T° alta).
- **Vigilancia:** L025 (dureza 52.07 > máx histórico 51.04; +2.5 HRC fuera de cadena).
- **Liberada:** L043 (SPE y Mahalanobis normales: extremo coherente).

## 4. Comparación de enfoques
| Método | Detecta | TP | FP | FN | Precision | Recall | F1 |
|---|---|---|---|---|---|---|---|
| Univariado ±3σ (Excel) | 16 | 15 | 1 | 15 | 0.94 | 0.50 | 0.65 |
| Autoencoder (Python) | 30 | 30 | 0 | 0 | 1.00 | 1.00 | 1.00 |
| Consenso (AE+Maha+SPE ≥2) | 30 | — | — | — | referencia | — | — |
Concordancia: Jaccard(AE,Uni)=0.50 · Kappa=0.40 → detectan defectos *diferentes*.

## 5. Métricas (justificación)
1. **Mahalanobis (χ²99%,6=16.81):** sensible a rupturas de covarianza. 2. **PCA SPE/Q + T²:**
separa anomalía de estructura (SPE) de extremo coherente (T²). 3. **Error AE por sensor:**
diagnóstico causal. 4. **Jaccard/Kappa:** comparabilidad de métodos. 5. **Sensibilidad de umbral:**
lista estable entre P97 y µ+3σ.

## 6. Conclusión
El filtro por columnas es ciego a la correlación: pierde el 50 % de los defectos reales
(L086–L100, patrón típico de tratamiento térmico/prensado desregulado) y alarma sobre piezas
buenas extremas (L043). El modelo multivariado aprende la cadena física del proceso y detecta
el 100 % con diagnóstico por sensor. Se recomienda esquema híbrido de dos capas y reclasificar
L025 con inspección metalográfica.